# Broca v2 — Pre-trained Encoder + EAM + Pre-trained Decoder

**The bottleneck in v1 was the encoder and decoder** — tiny GRUs trained from scratch on 1M chars.

v2 replaces both with pre-trained models:
- **Encoder:** `all-MiniLM-L6-v2` (22M params, frozen) — already understands language, outputs 384d unit vectors
- **Decoder:** GPT-2 small (124M params, frozen) — already knows how to generate text
- **Adapter:** Thin projection (424K params, **only trainable component**) — converts 384d thought → prefix tokens for GPT-2
- **EAM:** HeatherDB-compatible, 384d, 2000 locations

```
text → [sentence-transformer] → 384d → [EAM] → 384d → [adapter] → prefix → [GPT-2] → text
          22M (frozen)           disk read          424K (trained)     124M (frozen)
```

Only the adapter is trained. Everything else is off-the-shelf.

In [ ]:
!pip install sentence-transformers -q

In [ ]:
import math
import json
import urllib.request
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

## Load Pre-trained Models

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Encoder: text → 384d unit vector (frozen)
encoder = SentenceTransformer('all-MiniLM-L6-v2', device=str(device))
for p in encoder.parameters():
    p.requires_grad_(False)

# Decoder: GPT-2 small (frozen)
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
tokenizer.pad_token = tokenizer.eos_token
gpt2 = GPT2LMHeadModel.from_pretrained('gpt2').to(device)
for p in gpt2.parameters():
    p.requires_grad_(False)
gpt2.eval()

GPT2_DIM = gpt2.config.n_embd   # 768
THOUGHT_DIM = 384
GPT2_VOCAB = gpt2.config.vocab_size  # 50257

print(f"Encoder: {sum(p.numel() for p in encoder.parameters()):,} params (frozen)")
print(f"Decoder: {sum(p.numel() for p in gpt2.parameters()):,} params (frozen)")
print(f"Thought dim: {THOUGHT_DIM}, GPT-2 dim: {GPT2_DIM}")

## EAM + Adapter

In [ ]:
class EAMLayer(nn.Module):
    def __init__(self, num_locations=2000, dim=384, k=20, beta=5.0, t_max=3, eta=0.01):
        super().__init__()
        self.num_locations = num_locations
        self.dim = dim
        self.k = k
        self.beta = beta
        self.t_max = t_max
        self.eta = eta
        addresses = F.normalize(torch.randn(num_locations, dim), dim=1)
        self.register_buffer("addresses", addresses)
        self.register_buffer("counters", torch.zeros(num_locations, dim))
        self.register_buffer("write_counts", torch.zeros(num_locations))

    def read(self, query):
        squeeze = query.dim() == 1
        if squeeze:
            query = query.unsqueeze(0)
        wc = self.write_counts.clamp(min=1e-12).unsqueeze(1)
        raw_patterns = self.counters / wc
        has_writes = self.write_counts > 0
        patterns = F.normalize(raw_patterns, dim=1) * has_writes.float().unsqueeze(1)
        xi = F.normalize(query, dim=1)
        k = min(self.k, int(has_writes.sum().item()))
        if k == 0:
            return xi.squeeze(0) if squeeze else xi
        for _ in range(self.t_max):
            sims = torch.mm(xi, self.addresses.t())
            topk_sims, topk_idx = torch.topk(sims, k, dim=1)
            alpha = F.softmax(topk_sims * self.beta, dim=1)
            idx_expanded = topk_idx.unsqueeze(-1).expand(-1, -1, self.dim)
            topk_patterns = patterns.unsqueeze(0).expand(xi.shape[0], -1, -1)
            topk_patterns = torch.gather(topk_patterns, 1, idx_expanded)
            xi = (alpha.unsqueeze(-1) * topk_patterns).sum(dim=1)
            xi = F.normalize(xi, dim=1)
        return xi.squeeze(0) if squeeze else xi

    @torch.no_grad()
    def write(self, vector):
        if vector.dim() == 1:
            vector = vector.unsqueeze(0)
        vectors = F.normalize(vector, dim=1)
        k = min(self.k, self.num_locations)
        sims = torch.mm(vectors, self.addresses.t())
        topk_sims, topk_idx = torch.topk(sims, k, dim=1)
        weights = F.softmax(topk_sims * self.beta, dim=1)
        flat_idx = topk_idx.reshape(-1)
        flat_weights = weights.reshape(-1)
        vecs_expanded = vectors.unsqueeze(1).expand(-1, k, -1).reshape(-1, self.dim)
        weighted_vecs = flat_weights.unsqueeze(1) * vecs_expanded
        idx_for_counters = flat_idx.unsqueeze(1).expand(-1, self.dim)
        self.counters.scatter_add_(0, idx_for_counters, weighted_vecs)
        self.write_counts.scatter_add_(0, flat_idx, flat_weights)
        winners = topk_idx[:, 0]
        winner_addrs = self.addresses[winners]
        diff = vectors - winner_addrs
        self.addresses[winners] += self.eta * diff
        self.addresses[winners] = F.normalize(self.addresses[winners], dim=1)

    @torch.no_grad()
    def clear(self):
        self.counters.zero_()
        self.write_counts.zero_()

    def num_written(self):
        return int((self.write_counts > 0).sum().item())


class ThoughtAdapter(nn.Module):
    """Projects a 384d thought vector into GPT-2 prefix embeddings.

    This is the ONLY trainable component. 384d → 8 prefix tokens of 768d.
    GPT-2 attends to these prefix tokens as if they were context.
    """
    def __init__(self, thought_dim=384, gpt2_dim=768, n_prefix=8):
        super().__init__()
        self.n_prefix = n_prefix
        self.gpt2_dim = gpt2_dim
        self.project = nn.Sequential(
            nn.Linear(thought_dim, 128),
            nn.GELU(),
            nn.Linear(128, gpt2_dim * n_prefix),
        )

    def forward(self, thought):
        # thought: [B, 384] → prefix: [B, n_prefix, 768]
        return self.project(thought).view(-1, self.n_prefix, self.gpt2_dim)


N_PREFIX = 8
adapter = ThoughtAdapter(THOUGHT_DIM, GPT2_DIM, N_PREFIX).to(device)
eam = EAMLayer(num_locations=2000, dim=THOUGHT_DIM, k=20, beta=5.0, t_max=3).to(device)

trainable = sum(p.numel() for p in adapter.parameters())
print(f"Adapter: {trainable:,} trainable params")
print(f"EAM: 2000 locations x {THOUGHT_DIM}d")

## Prepare Data

1. Split Shakespeare into (context, target) pairs
2. Pre-encode all contexts with sentence-transformer → 384d vectors
3. Tokenize all targets with GPT-2 tokenizer

In [ ]:
def download_shakespeare():
    path = Path("tinyshakespeare.txt")
    if not path.exists():
        print("Downloading TinyShakespeare...")
        url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
        urllib.request.urlretrieve(url, str(path))
    return path.read_text()


class BrocaDataset(Dataset):
    def __init__(self, thoughts, target_ids, target_mask):
        self.thoughts = thoughts
        self.target_ids = target_ids
        self.target_mask = target_mask

    def __len__(self):
        return len(self.thoughts)

    def __getitem__(self, idx):
        return self.thoughts[idx], self.target_ids[idx], self.target_mask[idx]


# --- Config ---
CONTEXT_CHARS = 512     # input to sentence-transformer
TARGET_CHARS = 128      # text to predict
MAX_TARGET_TOKENS = 64  # max GPT-2 tokens per target
STRIDE = 64             # sliding window stride
BATCH_SIZE = 32
TRAIN_SPLIT = 0.9

# --- Load + split ---
text = download_shakespeare()
print(f"Corpus: {len(text):,} characters\n")

contexts, targets = [], []
for i in range(0, len(text) - CONTEXT_CHARS - TARGET_CHARS, STRIDE):
    contexts.append(text[i : i + CONTEXT_CHARS])
    targets.append(text[i + CONTEXT_CHARS : i + CONTEXT_CHARS + TARGET_CHARS])

print(f"Total pairs: {len(contexts):,}")

# --- Pre-encode contexts with sentence-transformer ---
print("Encoding contexts with sentence-transformer...")
thought_vectors = encoder.encode(
    contexts, batch_size=256, show_progress_bar=True,
    convert_to_tensor=True, device=str(device)
)
print(f"Thought vectors: {thought_vectors.shape}")

# --- Tokenize targets with GPT-2 ---
print("Tokenizing targets...")
target_enc = tokenizer(
    targets, max_length=MAX_TARGET_TOKENS, truncation=True,
    padding="max_length", return_tensors="pt"
)
target_ids = target_enc["input_ids"]
target_mask = target_enc["attention_mask"]
print(f"Target tokens: {target_ids.shape}")

# --- Train/val split ---
split = int(len(contexts) * TRAIN_SPLIT)
train_ds = BrocaDataset(thought_vectors[:split], target_ids[:split], target_mask[:split])
val_ds = BrocaDataset(thought_vectors[split:], target_ids[split:], target_mask[split:])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"\nTrain: {len(train_ds):,} ({len(train_loader)} batches)")
print(f"Val:   {len(val_ds):,}")

## Phase 1: Train Adapter

Only the adapter learns. GPT-2 and sentence-transformer are frozen.

The adapter converts 384d thought vectors into 8 prefix tokens.
GPT-2 attends to these prefixes and predicts the target text.

In [ ]:
optimizer = torch.optim.AdamW(adapter.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

EPOCHS = 50
PATIENCE = 7
best_val_loss = float("inf")
best_state = None
patience_counter = 0

print("--- Phase 1: Training adapter (prefix tuning) ---\n")

for epoch in range(EPOCHS):
    adapter.train()
    total_loss = 0.0
    num_batches = 0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:2d}", leave=False)
    for thoughts, tgt_ids, tgt_mask in pbar:
        thoughts = thoughts.to(device)
        tgt_ids = tgt_ids.to(device)
        tgt_mask = tgt_mask.to(device)
        B = thoughts.shape[0]

        optimizer.zero_grad()

        # Adapter: thought → prefix tokens
        prefix = adapter(thoughts)                           # [B, 8, 768]
        target_embeds = gpt2.transformer.wte(tgt_ids)        # [B, seq, 768]
        inputs_embeds = torch.cat([prefix, target_embeds], dim=1)

        # Attention mask: prefix always attended, target uses its mask
        attn_mask = torch.cat([
            torch.ones(B, N_PREFIX, device=device),
            tgt_mask.float()
        ], dim=1)

        # Labels: -100 for prefix + padding, real IDs for target tokens
        labels = tgt_ids.clone()
        labels[tgt_mask == 0] = -100
        labels = torch.cat([
            torch.full((B, N_PREFIX), -100, dtype=torch.long, device=device),
            labels
        ], dim=1)

        outputs = gpt2(inputs_embeds=inputs_embeds, attention_mask=attn_mask, labels=labels)
        loss = outputs.loss

        loss.backward()
        torch.nn.utils.clip_grad_norm_(adapter.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item()
        num_batches += 1
        pbar.set_postfix(loss=f"{loss.item():.3f}")

    # Validation
    adapter.eval()
    val_loss = 0.0
    val_batches = 0
    with torch.no_grad():
        for thoughts, tgt_ids, tgt_mask in val_loader:
            thoughts = thoughts.to(device)
            tgt_ids = tgt_ids.to(device)
            tgt_mask = tgt_mask.to(device)
            B = thoughts.shape[0]

            prefix = adapter(thoughts)
            target_embeds = gpt2.transformer.wte(tgt_ids)
            inputs_embeds = torch.cat([prefix, target_embeds], dim=1)
            attn_mask = torch.cat([
                torch.ones(B, N_PREFIX, device=device),
                tgt_mask.float()
            ], dim=1)
            labels = tgt_ids.clone()
            labels[tgt_mask == 0] = -100
            labels = torch.cat([
                torch.full((B, N_PREFIX), -100, dtype=torch.long, device=device),
                labels
            ], dim=1)

            outputs = gpt2(inputs_embeds=inputs_embeds, attention_mask=attn_mask, labels=labels)
            val_loss += outputs.loss.item()
            val_batches += 1

    avg_train = total_loss / num_batches
    avg_val = val_loss / val_batches
    train_ppl = math.exp(avg_train)
    val_ppl = math.exp(avg_val)
    lr = optimizer.param_groups[0]["lr"]

    scheduler.step(avg_val)

    if avg_val < best_val_loss:
        best_val_loss = avg_val
        best_state = {k: v.cpu().clone() for k, v in adapter.state_dict().items()}
        patience_counter = 0
        marker = " *"
    else:
        patience_counter += 1
        marker = ""

    print(f"Epoch {epoch+1:3d}: train_ppl={train_ppl:.2f} val_ppl={val_ppl:.2f} lr={lr:.1e}{marker}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping (best val_ppl={math.exp(best_val_loss):.2f})")
        break

adapter.load_state_dict(best_state)
adapter.to(device)
print(f"\nRestored best checkpoint (val_ppl={math.exp(best_val_loss):.2f})")

## Generate

In [ ]:
@torch.no_grad()
def generate(prompt, eam=None, max_tokens=150, temperature=0.8):
    """Encode prompt → [optional EAM read] → adapter prefix → GPT-2 generate."""
    adapter.eval()

    # Encode prompt with sentence-transformer
    thought = encoder.encode(prompt, convert_to_tensor=True, device=str(device))
    if thought.dim() == 1:
        thought = thought.unsqueeze(0)  # [1, 384]

    # Optional EAM read
    if eam is not None and eam.num_written() > 0:
        thought = eam.read(thought)

    # Adapter: thought → prefix
    prefix = adapter(thought)  # [1, 8, 768]

    # GPT-2: generate from prefix
    outputs = gpt2(inputs_embeds=prefix, use_cache=True)
    past = outputs.past_key_values
    next_logits = outputs.logits[:, -1, :] / temperature

    generated = []
    for _ in range(max_tokens):
        probs = torch.softmax(next_logits, dim=-1)
        next_token = torch.multinomial(probs, 1)  # [1, 1]
        generated.append(next_token.item())

        if next_token.item() == tokenizer.eos_token_id:
            break

        outputs = gpt2(input_ids=next_token, past_key_values=past, use_cache=True)
        past = outputs.past_key_values
        next_logits = outputs.logits[:, -1, :] / temperature

    return tokenizer.decode(generated, skip_special_tokens=True)

## Sample (no memory)

In [ ]:
prompts = [
    "ROMEO:",
    "To be, or not to be",
    "KING HENRY:",
    "And God said",
]

print("--- Samples (direct, no EAM) ---\n")
for p in prompts:
    print(f"Prompt: {p!r}")
    print(generate(p))
    print()

## Phase 2: Populate EAM

In [ ]:
eam.clear()

written = 0
with torch.no_grad():
    for thoughts, _, _ in tqdm(train_loader, desc="Writing to EAM"):
        thoughts = thoughts.to(device)
        eam.write(thoughts)
        written += thoughts.shape[0]

print(f"\nWrote {written:,} thought vectors to EAM")
print(f"Active locations: {eam.num_written()}/2000")

## Evaluation: Direct vs Memory

In [ ]:
adapter.eval()

for label, use_eam in [("Direct", False), ("Memory", True)]:
    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():
        for thoughts, tgt_ids, tgt_mask in val_loader:
            thoughts = thoughts.to(device)
            tgt_ids = tgt_ids.to(device)
            tgt_mask = tgt_mask.to(device)
            B = thoughts.shape[0]

            if use_eam:
                thoughts = eam.read(thoughts)

            prefix = adapter(thoughts)
            target_embeds = gpt2.transformer.wte(tgt_ids)
            inputs_embeds = torch.cat([prefix, target_embeds], dim=1)
            attn_mask = torch.cat([
                torch.ones(B, N_PREFIX, device=device),
                tgt_mask.float()
            ], dim=1)
            labels = tgt_ids.clone()
            labels[tgt_mask == 0] = -100
            labels = torch.cat([
                torch.full((B, N_PREFIX), -100, dtype=torch.long, device=device),
                labels
            ], dim=1)

            outputs = gpt2(inputs_embeds=inputs_embeds, attention_mask=attn_mask, labels=labels)
            total_loss += outputs.loss.item()
            total_batches += 1

    ppl = math.exp(total_loss / total_batches)
    print(f"{label:8s}: ppl={ppl:.2f}")

## Sample (with EAM)

In [ ]:
print("--- Samples (with EAM) ---\n")
for p in prompts:
    print(f"Prompt: {p!r}")
    print(generate(p, eam=eam))
    print()

## Export for HeatherDB

Downloads:
- `broca_v2.json` — EAM state (for `heather-fornix import`)
- `broca_v2_adapter.pt` — adapter weights (424K params)

The encoder (sentence-transformers) and decoder (GPT-2) are standard — load from HuggingFace at inference time.

In [ ]:
eam_cpu = EAMLayer(num_locations=2000, dim=THOUGHT_DIM)
eam_cpu.load_state_dict(eam.cpu().state_dict())

mem = eam_cpu
num_locs = mem.addresses.shape[0]
config = {
    "d": mem.dim, "l_0": num_locs, "l_max": num_locs * 2,
    "k": mem.k, "eta_0": mem.eta, "lambda": 0.9999,
    "eta_min": 0.001, "tau_split": 0.3, "tau_merge": 0.95,
    "gamma": 1.0, "tau_damp": 10.0, "tau_overload": 100.0,
    "beta": mem.beta, "t_max": mem.t_max, "epsilon": 1e-6,
}

addresses = mem.addresses.detach().double()
counters = mem.counters.detach().double()
write_counts = mem.write_counts.detach().double()

locations = [
    {"id": i, "address": addresses[i].tolist(),
     "counter": counters[i].tolist(),
     "write_count": float(write_counts[i])}
    for i in range(num_locs)
]

# Save adapter weights
torch.save(adapter.cpu().state_dict(), "broca_v2_adapter.pt")

export = {
    "config": config,
    "locations": locations,
    "controller_state": "broca_v2_adapter.pt",
    "prototypes": {},
    "model_config": {
        "encoder": "sentence-transformers/all-MiniLM-L6-v2",
        "decoder": "gpt2",
        "thought_dim": THOUGHT_DIM,
        "gpt2_dim": GPT2_DIM,
        "n_prefix": N_PREFIX,
        "adapter_hidden": 128,
    },
}

Path("broca_v2.json").write_text(json.dumps(export))

print(f"Exported: broca_v2.json ({Path('broca_v2.json').stat().st_size / 1024 / 1024:.1f} MB)")
print(f"Exported: broca_v2_adapter.pt ({Path('broca_v2_adapter.pt').stat().st_size / 1024:.0f} KB)")
print(f"\nAt inference time, load:")
print(f"  - Encoder: sentence-transformers/all-MiniLM-L6-v2 (from HuggingFace)")
print(f"  - Decoder: gpt2 (from HuggingFace)")
print(f"  - Adapter: broca_v2_adapter.pt ({trainable:,} params)")
print(f"  - EAM: heather-fornix import broca_v2.json")

In [ ]:
try:
    from google.colab import files
    files.download("broca_v2.json")
    files.download("broca_v2_adapter.pt")
except ImportError:
    print("Not in Colab — files saved locally.")